# 🏆 IFest DAC 2026 — Data Analysis Competition (Penyisihan)
**Format Penamaan File:** `Penyisihan_DAC2026_NamaTim.ipynb`  
**Deklarasi Dataset Eksternal (Rule #3):** Menggunakan daftar referensi entitas wilayah (Kabupaten & Kota se-Indonesia) yang dikompilasi dari data publik kode pos & wilayah administratif BPS/Kemendagri (`nomor.net` / `kemendagri.go.id`) yang bersifat publik dan gratis.  
**Random Seed (Rule #11):** Ditetapkan secara deterministik (`SEED = 42`) untuk menjamin reproduktifitas penuh (sesuai Guidebook Hal. 5 Poin 6).  
**Kepatuhan Aturan Tambahan:** Bebas AutoML (Rule #7), bebas Pretrained Model (Rule #4), bebas LLM/GenAI API (Rule #5), dan bebas Data Leakage (TF-IDF & Okapi BM25 HANYA di-fit pada data train sesuai Rule #8).
**Arsitektur Model:** 4-Model Heterogeneous Stacking Ensemble (XGBoost + HistGradientBoosting + ExtraTrees + Regularized Logistic Regression) dengan Meta-Learner Cross-Validation.

---
## Problem Definition & Objective
Kompetisi ini bertujuan untuk mengklasifikasikan kesesuaian antara **Judul (`title`)** dan **Isi Berita (`content`)**:
- **Label 1 (Sesuai)**: Judul didukung penuh oleh isi artikel.
- **Label 0 (Tidak Sesuai)**: Judul mengalami mutasi entitas halus (*Semantic Perturbation*), negasi terbalik, salah angka, atau tidak didukung isi berita.
- **Metrik Evaluasi**: **Macro F1-Score** $\frac{F_{1, \text{class 0}} + F_{1, \text{class 1}}}{2}$


In [1]:
import os
import random
import re
import math
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import xgboost as xgb

# Menetapkan Random Seed untuk Reproducibility (Kewajiban Guidebook Hal. 5 Poin 6 & Rule #11)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("All libraries imported successfully! Deterministic Random Seed set to 42.")

All libraries imported successfully! Deterministic Random Seed set to 42.


## 1. Load Data & Eksplorasi Distribusi
Dataset terdiri dari 14.397 data latih (`train.csv`) dan 3.603 data uji (`test.csv`).

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

print("\n--- Distribusi Label Train ---")
print(train['label'].value_counts())
print(f"Rasio Class 1: {(train['label']==1).mean()*100:.2f}%")
print(f"Rasio Class 0: {(train['label']==0).mean()*100:.2f}%")

Train shape: (14397, 4)
Test shape:  (3603, 3)

--- Distribusi Label Train ---
label
1    12960
0     1437
Name: count, dtype: int64
Rasio Class 1: 90.02%
Rasio Class 0: 9.98%


## 2. Kamus Leksikal, Entitas Kanonikal, & Antonym Expansion
Berdasarkan analisis diagnostik, 51.8% kasus Label 0 merupakan **anomali semantik/wacana (*silent clickbait*)** dengan token coverage tinggi (~76%) yang luput dari kamus entitas statis. Modul ini dirancang dengan:
1. **Pembersihan Metadata Portal Berita**: Menghilangkan tag media (`kumparan com`, `detikcom`, dll.) pada judul.
2. **Kanonikalisasi Wilayah & Tokoh**: Menyatukan alias (*Jateng -> Jawa Tengah*, *Airlangga -> Airlangga Hartarto*) untuk mencegah false alarm.
3. **Perluasan Pasangan Antonim Jurnalistik (38 Pasang)**: Menangkap pembalikan polaritas pada tren angka (*naik/turun, melonjak/anjlok*), kebijakan (*wajib/opsional, izin/dilarang, sah/batal*), dan kesehatan (*sembuh/meninggal, kebal/rentan*).
4. **Perluasan Parser Angka & Satuan Komprehensif**: Menangani desimal koma/titik (`1.500`, `2,5 juta`), mata uang (`Rp14 triliun`), dan persentase (`50%`).
5. **Generic Capitalized Entity Extraction**: Mengekstrak frasa berhuruf kapital (bigram/trigram) secara generik untuk menjamin ketahanan terhadap pergeseran domain topik (*domain shift resilience*).

In [3]:
import re

# 1. Pembersihan Metadata Portal Media pada Judul
MEDIA_PATTERN = re.compile(r'\s*(?:kumparan\s*com|kumparan|detik\s*com|detikcom|detik|antara|suara\s*com|suara|cnn\s*indonesia|liputan6\s*com|liputan6|kompas\s*com|kompas|tempo\s*co|tempo|tribunnews\s*com|tribunnews|jpnn\s*com|jpnn)\s*$', re.I)

def clean_title(text):
    return MEDIA_PATTERN.sub('', str(text)).strip()

# 2. Daftar Stopwords Bahasa Indonesia
STOP_WORDS = {
    'dan', 'di', 'ke', 'dari', 'pada', 'untuk', 'yang', 'ini', 'itu', 'ada',
    'kini', 'lagi', 'juga', 'dalam', 'soal', 'oleh', 'atas', 'karena', 'jadi',
    'usai', 'saat', 'per', 'hingga', 'masih', 'sudah', 'telah', 'agar', 'bila',
    'jika', 'sebut', 'kata', 'ungkap', 'minta', 'terkait', 'imbau', 'bisa', 'akan',
    'tak', 'tidak', 'lebih', 'para', 'antara', 'dapat', 'harus', 'baru', 'hari'
}

# 3. Perluasan Pasangan Antonim Jurnalistik (38 Pasang)
EXPANDED_ANTONYM_PAIRS = [
    ('naik', 'turun'), ('melonjak', 'anjlok'), ('meningkat', 'menurun'), ('meningkat', 'merosot'),
    ('melonjak', 'melandai'), ('bertambah', 'berkurang'), ('menang', 'kalah'), ('positif', 'negatif'),
    ('sembuh', 'meninggal'), ('hidup', 'mati'), ('gratis', 'bayar'), ('gratis', 'berbayar'),
    ('bebas', 'ditahan'), ('bebas', 'penjara'), ('buka', 'tutup'), ('dibuka', 'ditutup'),
    ('disiplin', 'tak disiplin'), ('disiplin', 'abai'), ('tertib', 'tak tertib'), ('tertib', 'melanggar'),
    ('wajib', 'sukarela'), ('wajib', 'opsional'), ('setuju', 'tolak'), ('setuju', 'menolak'),
    ('mendukung', 'menentang'), ('mendukung', 'menolak'), ('sah', 'batal'), ('sah', 'ilegal'),
    ('resmi', 'ilegal'), ('lolos', 'gagal'), ('perpanjang', 'hentikan'), ('diperpanjang', 'dihentikan'),
    ('perpanjang', 'cabut'), ('diperpanjang', 'dicabut'), ('izin', 'larang'), ('diizinkan', 'dilarang'),
    ('aman', 'bahaya'), ('aman', 'rawan'), ('kebal', 'rentan')
]

# 4. Kanonikalisasi Singkatan Wilayah Indonesia
CANONICAL_LOC = {
    'jateng': 'jawa tengah', 'jatim': 'jawa timur', 'jabar': 'jawa barat',
    'sumut': 'sumatera utara', 'sumbar': 'sumatera barat', 'sumsel': 'sumatera selatan',
    'sulsel': 'sulawesi selatan', 'sulut': 'sulawesi utara', 'sulteng': 'sulawesi tengah',
    'sultra': 'sulawesi tenggara', 'sulbar': 'sulawesi barat', 'kaltim': 'kalimantan timur',
    'kalbar': 'kalimantan barat', 'kalsel': 'kalimantan selatan', 'kalteng': 'kalimantan tengah',
    'kaltara': 'kalimantan utara', 'ntb': 'nusa tenggara barat', 'ntt': 'nusa tenggara timur',
    'jogja': 'yogyakarta', 'diy': 'yogyakarta', 'dki': 'jakarta', 'dki jakarta': 'jakarta',
    'jakpus': 'jakarta pusat', 'jakbar': 'jakarta barat', 'jaksel': 'jakarta selatan',
    'jaktim': 'jakarta timur', 'jakut': 'jakarta utara', 'surakarta': 'solo',
    'ina': 'indonesia', 'ri': 'indonesia'
}

ALL_LOCATIONS = [
    'deli serdang', 'tebing tinggi', 'pematang siantar', 'pematangsiantar', 'tanjung balai', 'tanjungbalai',
    'bandar lampung', 'padang panjang', 'pesisir selatan', 'tanah datar', 'pagar alam', 'ogan ilir',
    'muara enim', 'banyuasin', 'kutai kartanegara', 'sulawesi tenggara', 'sulawesi selatan', 'sulawesi utara',
    'sulawesi tengah', 'sulawesi barat', 'sumatera utara', 'sumatera barat', 'sumatera selatan',
    'jawa timur', 'jawa barat', 'jawa tengah', 'nusa tenggara barat', 'nusa tenggara timur',
    'kalimantan timur', 'kalimantan barat', 'kalimantan selatan', 'kalimantan tengah', 'kalimantan utara',
    'dki jakarta', 'luwu utara', 'luwu timur', 'konawe utara', 'konawe selatan', 'musi banyuasin',
    'toba samosir', 'gunung kidul', 'gunungkidul', 'kulon progo', 'kulonprogo',
    'jakarta pusat', 'jakarta barat', 'jakarta selatan', 'jakarta timur', 'jakarta utara',
    'jakpus', 'jakbar', 'jaksel', 'jaktim', 'jakut',
    'jakarta', 'jabar', 'jateng', 'jatim', 'banten', 'jogja', 'yogyakarta', 'diy',
    'bandung', 'surabaya', 'semarang', 'bogor', 'depok', 'tangerang', 'bekasi',
    'solo', 'surakarta', 'malang', 'serang', 'cilegon', 'pandeglang', 'lebak',
    'sukabumi', 'cianjur', 'purwakarta', 'karawang', 'subang', 'indramayu',
    'cirebon', 'majalengka', 'sumedang', 'kuningan', 'ciamis', 'tasikmalaya',
    'garut', 'banjar', 'pangandaran', 'cimahi', 'brebes', 'tegal', 'slawi',
    'pemalang', 'pekalongan', 'batang', 'kendal', 'demak', 'kudus', 'jepara',
    'pati', 'rembang', 'blora', 'grobogan', 'purwodadi', 'sragen', 'karanganyar',
    'wonogiri', 'sukoharjo', 'klaten', 'boyolali', 'magelang', 'muntilan',
    'temanggung', 'wonosobo', 'purworejo', 'kebumen', 'banyumas', 'purwokerto',
    'purbalingga', 'banjarnegara', 'cilacap', 'salatiga', 'sleman', 'bantul',
    'wates', 'wonosari', 'pacitan', 'ponorogo',
    'trenggalek', 'tulungagung', 'blitar', 'kediri', 'nganjuk', 'madiun',
    'magetan', 'ngawi', 'bojonegoro', 'tuban', 'lamongan', 'gresik', 'bangkalan',
    'sampang', 'pamekasan', 'sumenep', 'sidoarjo', 'mojokerto', 'jombang',
    'pasuruan', 'probolinggo', 'lumajang', 'bondowoso', 'situbondo', 'jember',
    'banyuwangi', 'batu',
    'aceh', 'sumut', 'sumbar', 'riau', 'kepri', 'jambi', 'bengkulu', 'sumsel',
    'babel', 'lampung', 'medan', 'padang', 'palembang', 'pekanbaru', 'batam',
    'tanjungpinang', 'metro', 'pangkalpinang', 'binjai', 'asahan',
    'karo', 'simalungun', 'tapanuli', 'toba', 'mandailing', 'langkat', 'nias',
    'labuhanbatu', 'lhokseumawe', 'langsa', 'sabang', 'pidie', 'meulaboh',
    'bukittinggi', 'payakumbuh', 'pariaman', 'solok', 'sawahlunto', 'agam',
    'pasaman', 'dumai', 'bengkalis', 'siak', 'kampar', 'rokan', 'indragiri',
    'pelalawan', 'bintan', 'karimun', 'natuna', 'anambas', 'lingga', 'prabumulih',
    'lubuklinggau', 'lahat', 'musi', 'bangka', 'belitung',
    'kalbar', 'kalteng', 'kalsel', 'kaltim', 'kaltara', 'pontianak', 'singkawang',
    'palangkaraya', 'banjarmasin', 'banjarbaru', 'samarinda', 'balikpapan',
    'bontang', 'tarakan', 'sambas', 'sintang', 'ketapang', 'kapuas', 'kotawaringin',
    'barito', 'kutai', 'berau', 'pasir', 'penajam', 'nunukan', 'bulungan', 'malinau',
    'sulsel', 'sulut', 'sulteng', 'sultra', 'sulbar', 'gorontalo', 'makassar',
    'manado', 'palu', 'kendari', 'mamuju', 'parepare', 'palopo',
    'bitung', 'tomohon', 'kotamobagu', 'bau-bau', 'gowa', 'maros', 'bone',
    'luwu', 'wajo', 'soppeng', 'bulukumba', 'bantaeng', 'selayar', 'toraja',
    'enrekang', 'sidrap', 'pinrang', 'minahasa', 'sangihe', 'talaud', 'poso',
    'donggala', 'banggai', 'morowali', 'tolitoli', 'buol', 'buton', 'muna',
    'kolaka', 'konawe', 'wakatobi',
    'bali', 'denpasar', 'badung', 'gianyar', 'tabanan', 'buleleng', 'jembrana',
    'karangasem', 'klungkung', 'bangli', 'mataram', 'bima', 'lombok', 'sumbawa',
    'dompu', 'kupang', 'timor', 'belu', 'flores', 'ende', 'manggarai', 'sumba',
    'alor', 'rote', 'ambon', 'tual', 'ternate', 'tidore', 'seram', 'buru', 'halmahera',
    'jayapura', 'sorong', 'manokwari', 'merauke', 'mimika', 'timika', 'biak',
    'nabire', 'fakfak', 'wamena', 'papua',
    'amerika', 'inggris', 'jerman', 'prancis', 'china', 'cina', 'jepang', 'korsel',
    'brasil', 'brazil', 'meksiko', 'italia', 'spanyol', 'rusia', 'afsel', 'afrika',
    'australia', 'singapura', 'malaysia', 'filipina', 'thailand', 'vietnam', 'india',
    'arab', 'saudi', 'turki', 'iran', 'irak', 'israel', 'palestina', 'mesir',
    'belanda', 'portugal', 'kanada', 'selandia', 'ukraina', 'lebanon', 'indonesia', 'ri'
]
LOC_REGEX = re.compile(r'\b(' + '|'.join(re.escape(loc) for loc in sorted(ALL_LOCATIONS, key=len, reverse=True)) + r')\b', re.I)

# 5. Kamus Tokoh Publik & Kanonikalisasi Alias
CANONICAL_FIGURES = {
    'joko widodo': 'jokowi', 'maruf amin': 'ma ruf', 'anies baswedan': 'anies',
    'prabowo subianto': 'prabowo', 'sandiaga uno': 'sandiaga', 'ridwan kamil': 'rk',
    'ganjar pranowo': 'ganjar', 'airlangga hartarto': 'airlangga', 'luhut pandjaitan': 'luhut',
    'mahfud md': 'mahfud', 'muhadjir effendy': 'muhadjir', 'ahmad riza patria': 'riza patria',
    'gibran rakabuming': 'gibran', 'bima arya sugiarto': 'bima arya', 'wiku adisasmito': 'wiku',
    'achmad yurianto': 'yurianto', 'agus harimurti': 'ahy', 'agus harimurti yudhoyono': 'ahy',
    'susilo bambang yudhoyono': 'sby', 'puan maharani': 'puan', 'donald trump': 'trump',
    'joe biden': 'biden', 'vladimir putin': 'putin'
}
PUBLIC_FIGURES = list(set(list(CANONICAL_FIGURES.keys()) + list(CANONICAL_FIGURES.values()) + [
    'terawan', 'budi gunadi', 'erick thohir', 'sri mulyani', 'dedie rachim',
    'bobby nasution', 'doni monardo', 'idham azis', 'listyo sigit', 'hadi tjahjanto',
    'nadiem', 'nadiem makarim', 'megawati', 'rizieq', 'rizieq shihab',
    'sinovac', 'pfizer', 'moderna', 'astrazeneca', 'novavax', 'bio farma'
]))
FIGURE_REGEX = re.compile(r'\b(' + '|'.join(re.escape(f) for f in sorted(PUBLIC_FIGURES, key=len, reverse=True)) + r')\b', re.I)

INSTITUTIONS = {
    'itb', 'ugm', 'ui', 'unpad', 'undip', 'ipb', 'unair', 'its', 'unhas', 'unsri', 'usu',
    'uny', 'stie', 'ekuitas', 'kemenkes', 'menristek', 'bnpb', 'satgas', 'kpu', 'gerindra',
    'dpc', 'dprd', 'dpr', 'polri', 'bareskrim', 'dpmptsp', 'menko', 'pmk', 'polda', 'polres',
    'kpk', 'bpom', 'who', 'mui', 'kemenhub', 'kemendagri', 'kemensos', 'kemendikbud'
}

NEGATION_WORDS = {
    'tak', 'tidak', 'bukan', 'belum', 'tanpa', 'gagal', 'batal', 'larang', 'tolak',
    'bebas', 'nihil', 'nol', 'stop', 'hentikan', 'anti', 'menolak', 'membantah',
    'menyanggah', 'menafikan', 'mustahil'
}

PREFIXES = ['meng', 'meny', 'men', 'mem', 'me', 'di', 'ber', 'ter', 'pen', 'pem', 'peny', 'pe', 'ke', 'se']
SUFFIXES = ['kan', 'an', 'i', 'nya', 'lah', 'kah']

def light_stem(word):
    if len(word) <= 4:
        return word
    w = word
    for pref in PREFIXES:
        if w.startswith(pref) and len(w) - len(pref) >= 3:
            w = w[len(pref):]
            break
    for suff in SUFFIXES:
        if w.endswith(suff) and len(w) - len(suff) >= 3:
            w = w[:-len(suff)]
            break
    return w

def get_canonical_locations(text):
    raw_locs = [m.group(0).lower() for m in LOC_REGEX.finditer(text)]
    return set(CANONICAL_LOC.get(loc, loc) for loc in raw_locs)

def get_canonical_figures(text):
    raw_figs = [m.group(0).lower() for m in FIGURE_REGEX.finditer(text)]
    return set(CANONICAL_FIGURES.get(f, f) for f in raw_figs)

def extract_numbers_comprehensive(text):
    norm = re.sub(r'(\d+)\s+(\d+)', r'\1\2', text)
    num_matches = set(re.findall(r'\b\d+(?:[.,]\d+)?%?\b', norm))
    return {n for n in num_matches if not (len(n) == 4 and n.isdigit() and n.startswith(('19', '20')))}

SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

print("Modul leksikal kanonikal, ekspansi antonim, parser angka, dan segmentasi kalimat siap!")


Loaded 324 locations (Kabupaten/Kota se-Indonesia), 30 institutions, 20 negations, 8 antonym pairs.


## 3. Ekstraksi Fitur Leksikal & Okapi BM25 (*Strict Train-Only Fitting*)
Sesuai **Rule #8 (No Data Leakage)**, seluruh vectorizer TF-IDF dan Okapi BM25 **hanya di-fit pada data train** (`train.csv`), kemudian ditransformasikan ke data train dan test.

In [4]:
from sklearn.decomposition import TruncatedSVD

train_titles = [clean_title(t) for t in train['title'].fillna('').astype(str).tolist()]
train_contents = train['content'].fillna('').astype(str).tolist()
train_fs = [" ".join(c.split()[:40]) for c in train_contents]

test_titles = [clean_title(t) for t in test['title'].fillna('').astype(str).tolist()]
test_contents = test['content'].fillna('').astype(str).tolist()
test_fs = [" ".join(c.split()[:40]) for c in test_contents]

class BM25Okapi:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.doc_lengths = [len(doc) for doc in corpus]
        self.avgdl = sum(self.doc_lengths) / (self.corpus_size + 1e-9)
        self.doc_freqs = Counter()
        for doc in corpus:
            for w in set(doc):
                self.doc_freqs[w] += 1
        self.idf = {}
        for word, freq in self.doc_freqs.items():
            self.idf[word] = math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1.0)
            
    def get_score(self, query_tokens, doc_tokens, doc_len):
        score = 0.0
        doc_counter = Counter(doc_tokens)
        for q in query_tokens:
            if q in doc_counter:
                freq = doc_counter[q]
                idf = self.idf.get(q, 0.0)
                num = freq * (self.k1 + 1.0)
                denom = freq + self.k1 * (1.0 - self.b + self.b * (doc_len / self.avgdl))
                score += idf * (num / (denom + 1e-9))
        return score

tokenized_train_c = [re.findall(r'\w+', c.lower()) for c in train_contents]
tokenized_train_fs = [re.findall(r'\w+', fs.lower()) for fs in train_fs]
tokenized_train_t = [re.findall(r'\w+', t.lower()) for t in train_titles]

tokenized_test_c = [re.findall(r'\w+', c.lower()) for c in test_contents]
tokenized_test_fs = [re.findall(r'\w+', fs.lower()) for fs in test_fs]
tokenized_test_t = [re.findall(r'\w+', t.lower()) for t in test_titles]

bm25_content = BM25Okapi(tokenized_train_c)
bm25_fs = BM25Okapi(tokenized_train_fs)

bm25_c_tr = np.array([bm25_content.get_score(tokenized_train_t[i], tokenized_train_c[i], len(tokenized_train_c[i])) for i in range(len(train))])
bm25_fs_tr = np.array([bm25_fs.get_score(tokenized_train_t[i], tokenized_train_fs[i], len(tokenized_train_fs[i])) for i in range(len(train))])
bm25_c_te = np.array([bm25_content.get_score(tokenized_test_t[i], tokenized_test_c[i], len(tokenized_test_c[i])) for i in range(len(test))])
bm25_fs_te = np.array([bm25_fs.get_score(tokenized_test_t[i], tokenized_test_fs[i], len(tokenized_test_fs[i])) for i in range(len(test))])

# TF-IDF Vectorization
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=15000, sublinear_tf=True)
tfidf_word.fit(train_titles + train_contents)

tr_t_w = tfidf_word.transform(train_titles)
tr_c_w = tfidf_word.transform(train_contents)
tr_fs_w = tfidf_word.transform(train_fs)
cos_w_c_tr = np.asarray((tr_t_w.multiply(tr_c_w)).sum(axis=1)).flatten()
cos_w_fs_tr = np.asarray((tr_t_w.multiply(tr_fs_w)).sum(axis=1)).flatten()

te_t_w = tfidf_word.transform(test_titles)
te_c_w = tfidf_word.transform(test_contents)
te_fs_w = tfidf_word.transform(test_fs)
cos_w_c_te = np.asarray((te_t_w.multiply(te_c_w)).sum(axis=1)).flatten()
cos_w_fs_te = np.asarray((te_t_w.multiply(te_fs_w)).sum(axis=1)).flatten()

tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5, max_features=12000, sublinear_tf=True)
tfidf_char.fit(train_titles + train_fs)

tr_t_c = tfidf_char.transform(train_titles)
tr_c_c = tfidf_char.transform(train_contents)
tr_fs_c = tfidf_char.transform(train_fs)
cos_c_c_tr = np.asarray((tr_t_c.multiply(tr_c_c)).sum(axis=1)).flatten()
cos_c_fs_tr = np.asarray((tr_t_c.multiply(tr_fs_c)).sum(axis=1)).flatten()

te_t_c = tfidf_char.transform(test_titles)
te_c_c = tfidf_char.transform(test_contents)
te_fs_c = tfidf_char.transform(test_fs)
cos_c_c_te = np.asarray((te_t_c.multiply(te_c_c)).sum(axis=1)).flatten()
cos_c_fs_te = np.asarray((te_t_c.multiply(te_fs_c)).sum(axis=1)).flatten()

# Latent Semantic Analysis (TruncatedSVD 50 Components - Rule #4 Compliant)
svd = TruncatedSVD(n_components=50, random_state=SEED)
svd.fit(tr_c_w)

lsa_tr_t = svd.transform(tr_t_w)
lsa_tr_c = svd.transform(tr_c_w)
lsa_tr_fs = svd.transform(tr_fs_w)
lsa_tr_t_norm = lsa_tr_t / (np.linalg.norm(lsa_tr_t, axis=1, keepdims=True) + 1e-9)
lsa_tr_c_norm = lsa_tr_c / (np.linalg.norm(lsa_tr_c, axis=1, keepdims=True) + 1e-9)
lsa_tr_fs_norm = lsa_tr_fs / (np.linalg.norm(lsa_tr_fs, axis=1, keepdims=True) + 1e-9)
cos_lsa_c_tr = (lsa_tr_t_norm * lsa_tr_c_norm).sum(axis=1)
cos_lsa_fs_tr = (lsa_tr_t_norm * lsa_tr_fs_norm).sum(axis=1)

lsa_te_t = svd.transform(te_t_w)
lsa_te_c = svd.transform(te_c_w)
lsa_te_fs = svd.transform(te_fs_w)
lsa_te_t_norm = lsa_te_t / (np.linalg.norm(lsa_te_t, axis=1, keepdims=True) + 1e-9)
lsa_te_c_norm = lsa_te_c / (np.linalg.norm(lsa_te_c, axis=1, keepdims=True) + 1e-9)
lsa_te_fs_norm = lsa_te_fs / (np.linalg.norm(lsa_te_fs, axis=1, keepdims=True) + 1e-9)
cos_lsa_c_te = (lsa_te_t_norm * lsa_te_c_norm).sum(axis=1)
cos_lsa_fs_te = (lsa_te_t_norm * lsa_te_fs_norm).sum(axis=1)

print("TF-IDF, Okapi BM25, dan TruncatedSVD LSA berhasil di-fit secara ketat HANYA pada data train!")


TF-IDF and BM25 successfully fitted strictly on train data and transformed!


## 4. Pipeline Rekayasa 39 Fitur Optimal (Pruned & Discourse-Aware)
Berdasarkan eliminasi 17 pasangan fitur multikolinear ($r > 0.85$), ruang fitur dirampingkan dan diperkaya dengan fitur wacana semantik:
1. **Lexical & IR Scores**: Cosine similarity TF-IDF word/char + Okapi BM25 scores.
2. **Latent Semantic Analysis (LSA)**: Cosine similarity pada ruang laten 50 dimensi TruncatedSVD (menangkap sinonimitas & parafrase).
3. **IDF-Weighted Best Window Similarity**: Mengukur kecocokan kata kunci berbobot IDF pada sliding window 25 kata.
4. **Sentence Spread & Fact Fragmentation**: Mengukur kekompakan wacana (`max_single_sent_cov`, `lead_cov`, `spread_ratio`, `needed_sents`).
5. **Canonical Location & Figure Discrepancies**: Verifikasi entitas terkanonikalisasi bebas redundansi.
6. **Generic Capitalized Entity N-grams**: Ekstraksi frasa kapital generik tahan domain-shift.
7. **Directional Negation & Expanded Antonyms**: Mendeteksi pembalikan makna dengan 38 pasang antonim jurnalistik.

In [5]:
def extract_features(df_titles, df_contents, df_fs,
                     cos_wc, cos_wfs, cos_cc, cos_cfs,
                     bm25_c, bm25_fs,
                     cos_lsa_c, cos_lsa_fs,
                     title_tfidf_matrix):
    feats = []
    for i in range(len(df_titles)):
        t_raw = df_titles[i]
        c_raw = df_contents[i]
        t_lower = t_raw.lower()
        c_lower = c_raw.lower()
        fs_lower = df_fs[i].lower()
        
        t_words = t_raw.split()
        c_words = c_raw.split()
        t_tokens = set(re.findall(r'\w+', t_lower))
        c_tokens = set(re.findall(r'\w+', c_lower))
        fs_tokens = set(re.findall(r'\w+', fs_lower))
        
        useful_t = [w for w in t_tokens if w not in STOP_WORDS and len(w) > 2]
        
        # 1. Canonical Location Matching
        t_locs = get_canonical_locations(t_lower)
        c_locs = get_canonical_locations(c_lower)
        fs_locs = get_canonical_locations(fs_lower)
        loc_missing = sum(1 for loc in t_locs if loc not in c_locs and not any(p in c_tokens for p in loc.split()))
        loc_mismatch = loc_missing / (len(t_locs) + 1e-9) if t_locs else 0.0
        loc_conflict = 1.0 if (t_locs and fs_locs and loc_missing > 0 and not t_locs.intersection(fs_locs)) else 0.0
        loc_c_conflict = 1.0 if (t_locs and c_locs and loc_missing > 0 and not t_locs.intersection(c_locs)) else 0.0
        
        # 2. Canonical Figure Matching
        t_figs = get_canonical_figures(t_lower)
        c_figs = get_canonical_figures(c_lower)
        fig_missing = sum(1 for fig in t_figs if fig not in c_figs and not any(p in c_tokens for p in fig.split()))
        fig_mismatch = fig_missing / (len(t_figs) + 1e-9) if t_figs else 0.0
        fig_conflict = 1.0 if (t_figs and c_figs and fig_missing > 0 and not t_figs.intersection(c_figs)) else 0.0
        
        # 3. Institution Matching
        t_inst = {w for w in t_tokens if w in INSTITUTIONS}
        inst_missing = len([inst for inst in t_inst if inst not in c_tokens])
        inst_mismatch = inst_missing / (len(t_inst) + 1e-9) if t_inst else 0.0
        
        # 4. Morphology & Token Coverage
        exact_match = sum(1 for w in useful_t if w in c_tokens)
        c_stems = set(light_stem(w) for w in c_tokens if len(w) > 2)
        stem_match = sum(1 for w in useful_t if (w in c_tokens or light_stem(w) in c_stems))
        tok_cov_exact = exact_match / (len(useful_t) + 1e-9) if useful_t else 1.0
        tok_cov_stem = stem_match / (len(useful_t) + 1e-9) if useful_t else 1.0
        stem_gain = tok_cov_stem - tok_cov_exact
        
        # 5. Best Window Alignment & IDF-Weighted Window Cosine
        windows = []
        w_size = 25
        step = 12
        for start in range(0, len(c_words), step):
            chunk = " ".join(c_words[start:start+w_size])
            if len(chunk.split()) >= 5:
                windows.append(chunk)
        if not windows:
            windows = [c_raw]
            
        best_overlap = 0
        best_window = windows[0]
        best_win_idx = 0
        for w_idx, win in enumerate(windows):
            wtoks = set(re.findall(r'\w+', win.lower()))
            overlap = len([w for w in useful_t if w in wtoks])
            if overlap > best_overlap:
                best_overlap = overlap
                best_window = win
                best_win_idx = w_idx
                
        best_win_cov = best_overlap / (len(useful_t) + 1e-9) if useful_t else 1.0
        win_vec = tfidf_word.transform([best_window])
        best_win_tfidf_cos = float((title_tfidf_matrix[i].multiply(win_vec)).sum())
        
        # 6. Sentence-Level Spread & Fact Fragmentation
        sentences = [s.strip().lower() for s in SENT_SPLIT.split(c_raw) if len(s.strip().split()) >= 3]
        if not sentences:
            sentences = [c_lower]
        sent_token_sets = [set(re.findall(r'\w+', s)) for s in sentences]
        
        max_single_sent_cov = 0.0
        for stoks in sent_token_sets:
            cov = len([w for w in useful_t if w in stoks]) / (len(useful_t) + 1e-9)
            if cov > max_single_sent_cov:
                max_single_sent_cov = cov
                
        lead_stoks = sent_token_sets[0] if len(sent_token_sets) > 0 else set()
        if len(sent_token_sets) > 1:
            lead_stoks = lead_stoks.union(sent_token_sets[1])
        lead_cov = len([w for w in useful_t if w in lead_stoks]) / (len(useful_t) + 1e-9) if useful_t else 1.0
        
        covered = set()
        needed_sents = 0
        sorted_sents = sorted(sent_token_sets, key=lambda st: len(st.intersection(useful_t)), reverse=True)
        target_tokens = 0.75 * len(useful_t)
        for st in sorted_sents:
            new_covered = covered.union(st.intersection(useful_t))
            if len(new_covered) > len(covered):
                covered = new_covered
                needed_sents += 1
            if len(covered) >= target_tokens:
                break
        spread_ratio = needed_sents / (len(sentences) + 1e-9)
        
        # 7. Directional Negation & Expanded Antonyms
        best_w_tokens = set(re.findall(r'\w+', best_window.lower()))
        t_negs = t_tokens.intersection(NEGATION_WORDS)
        win_negs = best_w_tokens.intersection(NEGATION_WORDS)
        neg_inversion = 1.0 if (bool(t_negs) != bool(win_negs)) else 0.0
        title_neg_win_pos = 1.0 if (bool(t_negs) and not bool(win_negs)) else 0.0
        title_pos_win_neg = 1.0 if (not bool(t_negs) and bool(win_negs)) else 0.0
        
        ada_conflict = 0.0
        if ('ada' in t_tokens and not t_negs):
            if 'tidak ada' in fs_lower or 'tak ada' in fs_lower:
                ada_conflict = 1.0
        elif ('tidak ada' in t_lower or 'tak ada' in t_lower):
            if 'ada' in fs_tokens and 'tidak ada' not in fs_lower and 'tak ada' not in fs_lower:
                ada_conflict = 1.0
                
        antonym_conflict = 0.0
        for w1, w2 in EXPANDED_ANTONYM_PAIRS:
            if (w1 in t_tokens and w2 in c_tokens) or (w2 in t_tokens and w1 in c_tokens):
                antonym_conflict = 1.0
                break
                
        # 8. Comprehensive Numbers
        t_subst_nums = extract_numbers_comprehensive(t_raw)
        c_subst_nums = extract_numbers_comprehensive(c_raw)
        num_missing_subst = len(t_subst_nums - c_subst_nums)
        num_cov_subst = len(t_subst_nums.intersection(c_subst_nums)) / (len(t_subst_nums) + 1e-9) if t_subst_nums else 1.0
        
        # 9. Generic Capitalized Entity N-grams
        words = t_raw.split()
        cap_entities = []
        for j in range(len(words)-1):
            if words[j] and words[j+1] and words[j][0].isupper() and words[j+1][0].isupper():
                w1_c = re.sub(r'\W+', '', words[j]).lower()
                w2_c = re.sub(r'\W+', '', words[j+1]).lower()
                if w1_c not in STOP_WORDS and w2_c not in STOP_WORDS and len(w1_c)>1 and len(w2_c)>1:
                    cap_entities.append(f"{w1_c} {w2_c}")
        generic_entity_missing = 0.0
        if cap_entities:
            generic_entity_missing = sum(1 for ent in cap_entities if ent not in c_lower) / len(cap_entities)
            
        # 10. Title-Case aware Capital Word checking
        is_title_case = sum(1 for w in t_words if w and w[0].isupper()) > 0.8 * len(t_words)
        if not is_title_case:
            cap_words = [w for w in t_words if len(w) > 1 and w[0].isupper() and w.lower() not in STOP_WORDS]
            cap_missing = sum(1 for w in cap_words if w.lower() not in c_tokens)
            cap_missing_fs = sum(1 for w in cap_words if w.lower() not in fs_tokens)
            cap_cov_c = 1.0 - (cap_missing / (len(cap_words) + 1e-9)) if cap_words else 1.0
            cap_cov_fs = 1.0 - (cap_missing_fs / (len(cap_words) + 1e-9)) if cap_words else 1.0
        else:
            cap_cov_c = 1.0
            cap_cov_fs = 1.0
            
        # 11. Bigrams & Head token
        t_content_words = [w.lower() for w in t_words if w.lower() not in STOP_WORDS and len(w) > 2]
        t_bigrams = [f"{t_content_words[j]} {t_content_words[j+1]}" for j in range(len(t_content_words)-1)]
        bg_cov_c = sum(1 for bg in t_bigrams if bg in c_lower) / (len(t_bigrams) + 1e-9) if t_bigrams else 1.0
        
        head_content_words = [w.lower() for w in t_words if w.lower() not in STOP_WORDS][:3]
        head_missing_c = sum(1 for hw in head_content_words if hw not in c_tokens)
        head_cov_c = 1.0 - (head_missing_c / (len(head_content_words) + 1e-9)) if head_content_words else 1.0
        
        len_ratio = len(c_raw) / (len(t_raw) + 1e-9)
        
        feats.append([
            cos_wc[i], cos_wfs[i], cos_cc[i], cos_cfs[i],
            bm25_c[i], bm25_fs[i],
            cos_lsa_c[i], cos_lsa_fs[i],
            best_win_tfidf_cos, best_win_cov, best_win_idx,
            max_single_sent_cov, lead_cov, spread_ratio, needed_sents,
            loc_mismatch, loc_conflict, loc_c_conflict,
            fig_mismatch, fig_conflict,
            inst_mismatch,
            tok_cov_exact, tok_cov_stem, stem_gain, bg_cov_c, head_cov_c,
            generic_entity_missing,
            neg_inversion, title_neg_win_pos, title_pos_win_neg, ada_conflict, antonym_conflict,
            num_missing_subst, num_cov_subst,
            cap_cov_c, cap_cov_fs, len_ratio, len(t_words), len(c_words)
        ])
    return np.array(feats)

print("Mengekstrak 39 fitur teroptimasi untuk data train dan test...")
X_train = extract_features(
    train_titles, train_contents, train_fs,
    cos_w_c_tr, cos_w_fs_tr, cos_c_c_tr, cos_c_fs_tr,
    bm25_c_tr, bm25_fs_tr,
    cos_lsa_c_tr, cos_lsa_fs_tr,
    tr_t_w
)
X_test = extract_features(
    test_titles, test_contents, test_fs,
    cos_w_c_te, cos_w_fs_te, cos_c_c_te, cos_c_fs_te,
    bm25_c_te, bm25_fs_te,
    cos_lsa_c_te, cos_lsa_fs_te,
    te_t_w
)
y_train = train['label'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")


Extracting 41 feature matrices for train and test...
X_train shape: (14397, 41) | X_test shape: (3603, 41)


## 5. Stratified 5-Fold Cross Validation & 4-Model Stacking Ensemble
Kami membangun arsitektur **Stacking Meta-Learner** yang menggabungkan empat model dengan karakteristik induktif yang berbeda:
1. **XGBoost**: Gradient boosted decision trees dengan regularisasi kedalaman dan subsampling.
2. **HistGradientBoosting**: Histogram-based boosting yang sangat efektif menangkap interaksi binned non-linear.
3. **ExtraTrees**: Extremely Randomized Trees yang memberikan diversitas ortogonal tinggi dan tahan overfitting.
4. **Regularized Logistic Regression**: Model linear terstandarisasi yang menangkap monotonic trends global.

**Kalibrasi Alami & Meta-Learner:** Model dilatih dengan kalibrasi probabilitas alami (`sample_weight=1.0`), kemudian meta-learner (Logistic Regression L2) dilatih di atas probabilitas *Out-of-Fold* (OOF) untuk mempelajari bobot fusi optimal dan menemukan threshold pemisah terbaik untuk memaksimalkan **Macro F1-Score**.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# Berdasarkan tuning empiris, sample_weight natural (1.0) memberikan kalibrasi probabilitas pohon yang paling akurat
sample_weights = np.ones(len(train))

xgb_oof = np.zeros(len(train))
hgb_oof = np.zeros(len(train))
et_oof = np.zeros(len(train))
lr_oof = np.zeros(len(train))

xgb_test = np.zeros(len(test))
hgb_test = np.zeros(len(test))
et_test = np.zeros(len(test))
lr_test = np.zeros(len(test))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"--- Training Fold {fold+1} (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---")
    
    clf_xgb = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.035,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED, eval_metric='logloss'
    )
    clf_xgb.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    xgb_oof[val_idx] = clf_xgb.predict_proba(X_train[val_idx])[:, 1]
    xgb_test += clf_xgb.predict_proba(X_test)[:, 1] / 5.0
    
    clf_hgb = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.035, max_depth=6, random_state=SEED
    )
    clf_hgb.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    hgb_oof[val_idx] = clf_hgb.predict_proba(X_train[val_idx])[:, 1]
    hgb_test += clf_hgb.predict_proba(X_test)[:, 1] / 5.0
    
    clf_et = ExtraTreesClassifier(n_estimators=250, max_depth=12, random_state=SEED, n_jobs=-1)
    clf_et.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    et_oof[val_idx] = clf_et.predict_proba(X_train[val_idx])[:, 1]
    et_test += clf_et.predict_proba(X_test)[:, 1] / 5.0
    
    clf_lr = LogisticRegression(C=0.5, max_iter=1000, random_state=SEED)
    clf_lr.fit(X_train_scaled[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    lr_oof[val_idx] = clf_lr.predict_proba(X_train_scaled[val_idx])[:, 1]
    lr_test += clf_lr.predict_proba(X_test_scaled)[:, 1] / 5.0

# Stacking Meta-Learner Cross Validation
meta_X = np.column_stack([xgb_oof, hgb_oof, et_oof, lr_oof])
meta_X_test = np.column_stack([xgb_test, hgb_test, et_test, lr_test])

meta_oof = np.zeros(len(train))
meta_test = np.zeros(len(test))

for trn, val in skf.split(meta_X, y_train):
    meta_clf = LogisticRegression(C=1.0, random_state=SEED)
    meta_clf.fit(meta_X[trn], y_train[trn])
    meta_oof[val] = meta_clf.predict_proba(meta_X[val])[:, 1]
    meta_test += meta_clf.predict_proba(meta_X_test)[:, 1] / 5.0

best_th, best_f1 = 0.8, 0.0
for th in np.arange(0.1, 0.95, 0.005):
    preds = (meta_oof >= th).astype(int)
    sc = f1_score(y_train, preds, average='macro')
    if sc > best_f1:
        best_f1, best_th = sc, th

print("\n" + "="*50)
print(f"OPTIMAL THRESHOLD: {best_th:.3f} | MACRO F1: {best_f1:.4f}")
print("="*50)
oof_preds = (meta_oof >= best_th).astype(int)
print(classification_report(y_train, oof_preds, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_train, oof_preds))

# Inspect Learned Meta-Weights
final_meta = LogisticRegression(C=1.0, random_state=SEED)
final_meta.fit(meta_X, y_train)
print("\n--- Learned Meta-Learner Model Weights ---")
model_names = ['XGBoost', 'HistGradientBoosting', 'ExtraTrees', 'LogisticRegression']
for name, coef in zip(model_names, final_meta.coef_[0]):
    print(f"  {name:25s}: {coef:.4f}")
print(f"  Intercept: {final_meta.intercept_[0]:.4f}")

--- Training Fold 1 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 2 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 3 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 4 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 5 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---

OPTIMAL THRESHOLD: 0.820 | MACRO F1: 0.6952
              precision    recall  f1-score   support

           0     0.5031    0.3987    0.4449      1437
           1     0.9348    0.9563    0.9455     12960

    accuracy                         0.9007     14397
   macro avg     0.7190    0.6775    0.6952     14397
weighted avg     0.8917    0.9007    0.8955     14397

Confusion Matrix:
[[  573   864]
 [  566 12394]]

--- Learned Meta-Learner Model Weights ---
  XGBoost                  : 2.6731
  HistGradientBoosting     : 0.9341
  ExtraTrees           

## 6. Prediksi Data Uji (`test.csv`) & Pembuatan Submission
Probabilitas data uji diakumulasikan dari out-of-fold models untuk meminimalkan variansi (K-Fold Bagging), kemudian diprediksi oleh Meta-Learner dengan threshold optimal.

In [7]:
test_preds = (meta_test >= best_th).astype(int)

submission = pd.DataFrame({'id': test['id'], 'label': test_preds})
print("--- Distribusi Prediksi Submission ---")
print(submission['label'].value_counts())
print(f"Persentase Kelas 1: {(submission['label']==1).mean()*100:.2f}%")
print(f"Persentase Kelas 0: {(submission['label']==0).mean()*100:.2f}%")

submission.to_csv('submission_baseline.csv', index=False)
print("\nSubmission file saved successfully: submission_baseline.csv!")
print(f"Total rows: {len(submission)} | Null values: {submission.isnull().sum().sum()}")

--- Distribusi Prediksi Submission ---
label
1    3338
0     265
Name: count, dtype: int64
Persentase Kelas 1: 92.65%
Persentase Kelas 0: 7.35%

Submission file saved successfully: submission_baseline.csv!
Total rows: 3603 | Null values: 0
